# 역전파 (1986) — 논문의 두 실험을 다시 돌려 본다

Rumelhart, Hinton, Williams 의 식 (1)-(9) 를 그대로 구현해, **노트의 「남는 질문」 네 가지에 답이 나오는지**를 본다.

| 실험 | 묻는 것 | 어느 질문에 닿는가 |
|---|---|---|
| A | 대칭 검출이 풀리는가. 1 : 2 : 4 구조와 부호 반대칭이 seed 를 바꿔도 다시 나오는가 | 질문 1 |
| B | 은닉 유닛을 2 개에서 늘리면 못 푸는 seed 비율이 달라지는가 | 질문 4 |
| C | 가속 항 α 를 빼면 얼마나 느려지는가 | 식 (9) 의 값어치 |
| D | 가족 트리에서 미학습 사실을 맞히는가. **논문의 설정값 그대로 1,500 훑기에 풀리는가** | 질문 2 |
| D-3 | 가중치 감쇠를 빼면 은닉 유닛의 축(국적 · 세대)이 덜 읽히는가 | 질문 3 |
| E | 층이 깊어질 때 층별 기울기가 몇 배씩 줄어드는가 | 노트 12 절의 0.25 이야기 |
| F | 퍼셉트론이 못 풀던 XOR 가 은닉층 둘로 풀리는가 | 퍼셉트론 노트북과 잇는 다리 |

**돌리는 법.** 위에서 아래로 모두 실행한다. CPU 만 쓰고 바깥 데이터를 읽지 않는다. 전체가 몇 분 안에 끝난다.
결과는 `results/` 에 CSV 로, 그림은 `figures/` 에 PNG 로 떨어지고, 맨 아래 칸이 요약을 찍는다.

**축소 규칙을 미리 적어 둔다.** 한 실험이 5 분을 넘으면 seed 수를 절반으로 줄인다. 훑기 상한은 줄이지 않는다 —
상한을 줄이면 「못 풀었다」가 「느렸다」와 섞인다.

In [ ]:
# ── 준비 ────────────────────────────────────────────────────────────────
import os, sys, time, itertools, platform
import numpy as np
import pandas as pd

NOTEBOOK = "backprop_1986.ipynb"
RUN_ID   = time.strftime("%Y%m%d_%H%M%S")
HERE     = os.getcwd()
RESULTS  = os.path.join(HERE, "results")
FIGURES  = os.path.join(HERE, "figures")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

ENV = {
    "run_id": RUN_ID, "notebook": NOTEBOOK,
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "platform": platform.platform(), "cwd": HERE,
}
for k, v in ENV.items():
    print("%-10s %s" % (k, v))


def save(df, name):
    """결과 CSV 에 재현 정보를 열로 붙여 저장한다 (experiments/README.md 규칙 3)."""
    df = df.copy()
    for k in ("run_id", "notebook", "python", "numpy", "platform"):
        df[k] = ENV[k]
    path = os.path.join(RESULTS, name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("저장:", name, df.shape)
    return df

## 0. 식 (1)-(9) 구현

순방향은 식 (1)(2), 역방향은 식 (4)-(7), 갱신은 식 (8)(9) 다. 함수 하나에 식 번호를 주석으로 달아 둔다.

$$x_j = \sum_i y_i w_{ji} \quad (1) \qquad y_j = \frac{1}{1+e^{-x_j}} \quad (2) \qquad E = \tfrac{1}{2}\sum_c\sum_j (y_{j,c}-d_{j,c})^2 \quad (3)$$

$$\frac{\partial E}{\partial y_j} = y_j - d_j \;(4) \qquad
\frac{\partial E}{\partial x_j} = \frac{\partial E}{\partial y_j} y_j(1-y_j) \;(5) \qquad
\frac{\partial E}{\partial w_{ji}} = \frac{\partial E}{\partial x_j} y_i \;(6) \qquad
\frac{\partial E}{\partial y_i} = \sum_j \frac{\partial E}{\partial x_j} w_{ji} \;(7)$$

$$\Delta w = -\varepsilon \frac{\partial E}{\partial w} \;(8) \qquad
\Delta w(t) = -\varepsilon \frac{\partial E}{\partial w(t)} + \alpha\, \Delta w(t-1) \;(9)$$

In [ ]:
def logistic(x):
    return 1.0 / (1.0 + np.exp(-x))


def init_net(sizes, rng, lo=-0.3, hi=0.3):
    """논문은 대칭 검출에서 처음 가중치를 -0.3 과 0.3 사이의 균등 난수로 두었다.
    편향도 가중치의 하나이므로 같은 범위에서 뽑는다."""
    W = [rng.uniform(lo, hi, (sizes[i], sizes[i + 1])) for i in range(len(sizes) - 1)]
    b = [rng.uniform(lo, hi, (sizes[i + 1],)) for i in range(len(sizes) - 1)]
    return W, b


def forward(W, b, X):
    """식 (1)(2). 층마다의 출력을 모두 돌려준다 — 역방향에 그 값이 필요하다."""
    ys = [X]
    for Wi, bi in zip(W, b):
        ys.append(logistic(ys[-1] @ Wi + bi))
    return ys


def backward(W, ys, D, margin=None):
    """식 (4)-(7). margin 을 주면 이미 맞은 출력의 오차를 0 으로 친다
    (논문이 가족 트리에서 쓴 장치: 켜져야 할 것이 0.8 위, 꺼져야 할 것이 0.2 아래면 오차 0)."""
    gW = [None] * len(W)
    gb = [None] * len(W)
    Y = ys[-1]
    dEdy = Y - D                                          # 식 (4)
    if margin is not None:
        hi, lo = margin
        dEdy = np.where(np.where(D >= 0.5, Y > hi, Y < lo), 0.0, dEdy)
    for L in range(len(W) - 1, -1, -1):
        dEdx = dEdy * ys[L + 1] * (1.0 - ys[L + 1])       # 식 (5)
        gW[L] = ys[L].T @ dEdx                            # 식 (6)
        gb[L] = dEdx.sum(axis=0)
        dEdy = dEdx @ W[L].T                              # 식 (7)
    return gW, gb


def error(Y, D):
    """식 (3)."""
    return 0.5 * float(((Y - D) ** 2).sum())


def solved(Y, D, hi=0.8, lo=0.2):
    """논문이 가족 트리에서 쓴 판정을 그대로 가져와 「풀렸다」의 기준으로 삼는다.
    켜져야 할 출력이 hi 위, 꺼져야 할 출력이 lo 아래면 그 예를 맞힌 것으로 본다."""
    return np.all(np.where(D >= 0.5, Y > hi, Y < lo), axis=1)


def train(X, D, sizes, seed, eps=0.1, alpha=0.9, sweeps=20000, decay=0.0,
          margin=None, mask0=None, schedule=None, stop_when_solved=True):
    """훑기마다 기울기를 모아 한 번 갱신한다 (식 8, 9).

    schedule: t 를 받아 (eps, alpha) 를 돌려주는 함수. 주면 eps/alpha 대신 쓴다.
    mask0:    첫 층의 연결을 막는 0/1 행렬. 가족 트리 망에서 사람과 관계를 따로 보내는 데 쓴다.
    돌려주는 것: dict"""
    rng = np.random.default_rng(seed)
    W, b = init_net(sizes, rng)
    if mask0 is not None:
        W[0] = W[0] * mask0
    vW = [np.zeros_like(w) for w in W]
    vb = [np.zeros_like(x) for x in b]
    first_solved = None
    for t in range(1, sweeps + 1):
        e, a = schedule(t) if schedule else (eps, alpha)
        ys = forward(W, b, X)
        gW, gb = backward(W, ys, D, margin=margin)
        for L in range(len(W)):
            vW[L] = -e * gW[L] + a * vW[L]                 # 식 (9)
            W[L] = W[L] + vW[L]
            vb[L] = -e * gb[L] + a * vb[L]
            b[L] = b[L] + vb[L]
            if decay:                                      # 가중치 감쇠
                W[L] = W[L] * (1.0 - decay)
                b[L] = b[L] * (1.0 - decay)
        if mask0 is not None:
            W[0] = W[0] * mask0
        if first_solved is None and solved(ys[-1], D).all():
            first_solved = t
            if stop_when_solved:
                break
    ys = forward(W, b, X)
    return dict(W=W, b=b, sweeps_run=t, first_solved=first_solved,
                converged=first_solved is not None,
                E=error(ys[-1], D), Y=ys[-1])

### 구현이 맞는지 먼저 확인한다 — 유한 차분

식 (4)-(7) 로 구한 $\partial E/\partial w$ 를, 가중치를 아주 조금 흔들어 $E$ 가 얼마나 변하는지로 다시 구해 견준다.
**두 값이 맞지 않으면 아래 실험은 모두 의미가 없다.**

In [ ]:
rng = np.random.default_rng(0)
Xc = rng.uniform(0, 1, (7, 4))
Dc = rng.integers(0, 2, (7, 3)).astype(float)
Wc, bc = init_net([4, 5, 3], rng)

ys = forward(Wc, bc, Xc)
gW, gb = backward(Wc, ys, Dc)

h = 1e-6
worst = 0.0
for L in range(len(Wc)):
    for _ in range(12):
        i = rng.integers(Wc[L].shape[0]); j = rng.integers(Wc[L].shape[1])
        Wp = [w.copy() for w in Wc]; Wp[L][i, j] += h
        Wm = [w.copy() for w in Wc]; Wm[L][i, j] -= h
        num = (error(forward(Wp, bc, Xc)[-1], Dc) - error(forward(Wm, bc, Xc)[-1], Dc)) / (2 * h)
        ana = gW[L][i, j]
        worst = max(worst, abs(num - ana) / max(1e-12, abs(num) + abs(ana)))
print("식 (4)-(7) 로 구한 기울기와 유한 차분의 상대 차이, 가장 큰 것: %.2e" % worst)
print("판정:", "맞다 (1e-6 아래)" if worst < 1e-6 else "다르다 — 아래 실험을 믿으면 안 된다")

## A. 대칭 검출 재현 (논문 그림 1)

입력 6 개의 이진 벡터가 가운데를 기준으로 대칭이면 1 을 내게 한다. 64 가지 입력 전부를 쓴다.
논문의 설정: 은닉 2 개, ε = 0.1, α = 0.9, 처음 가중치 -0.3 ~ 0.3, 훑기마다 갱신. 논문은 **1,425 훑기**가 걸렸다고 적었다.

**논문이 적지 않은 것이 하나 있다 — 언제 「풀렸다」고 보았는지다.** 여기서는 논문이 가족 트리에서 쓴 기준
(켜져야 할 것 0.8 위, 꺼져야 할 것 0.2 아래)을 그대로 빌려 쓴다. **이 기준은 내가 고른 것이다.**

- **H1**: 대부분의 seed 에서 풀리고, 걸린 훑기 수의 중앙값이 1,425 와 같은 자릿수다.
- **H2**: 풀린 seed 에서는 가운데를 기준으로 마주 보는 가중치가 크기가 같고 부호가 반대다.
- **H3**: 한쪽 절반의 세 크기 비가 1 : 2 : 4 다.
- **반대 방향**: H2 나 H3 가 일부 seed 에서만 나온다면, 논문의 그림 1 은 **여러 해 중 하나**이지 유일한 해가 아니다.

In [ ]:
# 대칭 검출 데이터
SYM_X = np.array(list(itertools.product([0, 1], repeat=6)), dtype=float)
SYM_D = np.array([[1.0 if np.array_equal(p, p[::-1]) else 0.0] for p in SYM_X])
print("입력", SYM_X.shape, " 대칭인 것", int(SYM_D.sum()), "개")

MAX_SWEEPS = 20000
SEEDS_A = 50


def symmetry_stats(W):
    """은닉 유닛의 가중치에서 부호 반대칭과 1 : 2 : 4 를 잰다.

    anti  : ||w + 뒤집은 w|| / ||w||. 0 이면 완전한 부호 반대칭이다.
    ratios: 위쪽 절반 세 개의 크기를 가장 작은 것으로 나눈 값 (오름차순)
    mirror: 두 은닉 유닛이 서로 부호만 반대인 정도"""
    w0 = W[0][:, 0]
    w1 = W[0][:, 1]
    anti = [float(np.linalg.norm(w + w[::-1]) / max(1e-12, np.linalg.norm(w))) for w in (w0, w1)]
    ratios = []
    for w in (w0, w1):
        half = np.sort(np.abs(w[:3]))
        ratios.append(half / max(1e-12, half[0]))
    mirror = float(np.linalg.norm(w0 + w1) / max(1e-12, np.linalg.norm(w0)))
    return anti, ratios, mirror


rows = []
t0 = time.time()
for s in range(SEEDS_A):
    r = train(SYM_X, SYM_D, [6, 2, 1], seed=s, eps=0.1, alpha=0.9, sweeps=MAX_SWEEPS)
    anti, ratios, mirror = symmetry_stats(r["W"])
    rows.append(dict(seed=s, converged=r["converged"], sweeps=r["first_solved"] or MAX_SWEEPS,
                     E=r["E"], max_sweeps=MAX_SWEEPS, hidden=2, eps=0.1, alpha=0.9,
                     anti_h1=anti[0], anti_h2=anti[1], mirror=mirror,
                     r1=ratios[0][0], r2=ratios[0][1], r3=ratios[0][2],
                     w_h1=";".join("%.2f" % v for v in r["W"][0][:, 0]),
                     bias_h1=r["b"][0][0], bias_out=float(r["b"][1][0]),
                     w_out=";".join("%.2f" % v for v in r["W"][1][:, 0])))
A = save(pd.DataFrame(rows), "A_symmetry.csv")
print("%.1f초" % (time.time() - t0))

ok = A[A.converged]
print()
print("풀린 seed %d / %d" % (len(ok), len(A)))
if len(ok):
    print("훑기  중앙값 %.0f   최소 %d   최대 %d   (논문은 1,425 라 적었다)"
          % (ok.sweeps.median(), ok.sweeps.min(), ok.sweeps.max()))
    print("부호 반대칭 지표  중앙값 %.4f  최대 %.4f   (0 이면 완전한 반대칭)"
          % (ok[["anti_h1", "anti_h2"]].median().mean(), ok[["anti_h1", "anti_h2"]].max().max()))
    print("두 은닉 유닛이 서로 반대인 정도  중앙값 %.4f" % ok.mirror.median())
    print("위쪽 절반 세 크기의 비  중앙값  %.2f : %.2f : %.2f   (논문의 그림 1 은 1 : 2 : 4)"
          % (ok.r1.median(), ok.r2.median(), ok.r3.median()))
print()
print("못 푼 seed 의 오차 E:", sorted(np.round(A[~A.converged].E.values, 3)))
print("(대칭인 입력이 8 개이므로 늘 0 을 내는 해의 오차는 0.5 x 8 = 4.0 이다)")

## B. 은닉 유닛을 늘리면 갇히는 일이 줄어드는가

논문은 국소 최소에 갇히는 것을 겪은 것이 **과제를 수행할 만큼만 연결이 있는 망**뿐이었고,
**연결을 몇 개 더 두면 차원이 늘어 벽을 돌아가는 길이 생긴다**고 적었다. 숫자는 붙어 있지 않다.

- **H1**: 은닉 유닛이 2 개일 때보다 3, 4, 6 개일 때 못 푸는 seed 비율이 낮다.
- **H1-a**: 비율이 같거나 오히려 높다면, 이 과제에서는 그 서술이 재현되지 않은 것이다. 그대로 적는다.

In [ ]:
HIDDEN_LIST = [2, 3, 4, 6]
SEEDS_B = 50

rows = []
t0 = time.time()
for h in HIDDEN_LIST:
    for s in range(SEEDS_B):
        r = train(SYM_X, SYM_D, [6, h, 1], seed=s, eps=0.1, alpha=0.9, sweeps=MAX_SWEEPS)
        rows.append(dict(hidden=h, seed=s, converged=r["converged"],
                         sweeps=r["first_solved"] or MAX_SWEEPS, E=r["E"],
                         max_sweeps=MAX_SWEEPS, n_weights=6 * h + h + h + 1))
# 대칭인 입력이 8 개이므로 늘 0 을 내는 해의 오차는 0.5 x 8 = 4.0 이다.
# 못 푼 seed 를 이 해에 갇힌 것과 그냥 기준에 못 미친 것으로 가른다 — 둘은 다른 이야기다.
ALWAYS_OFF_E = 0.5 * float(SYM_D.sum())
rows_df = pd.DataFrame(rows)
rows_df["stuck_all_off"] = (~rows_df.converged) & rows_df.E.between(ALWAYS_OFF_E * 0.98, ALWAYS_OFF_E * 1.02)
rows_df["slow_only"] = (~rows_df.converged) & (~rows_df.stuck_all_off)
Bdf = save(rows_df, "B_hidden.csv")
print("%.1f초" % (time.time() - t0))

print()
print("늘 0 을 내는 해의 오차 E = %.1f" % ALWAYS_OFF_E)
g = Bdf.groupby("hidden").agg(가중치_수=("n_weights", "first"),
                              푼_seed=("converged", "sum"),
                              전체=("seed", "count"),
                              훑기_중앙값=("sweeps", "median"),
                              늘0을_내는_해=("stuck_all_off", "sum"),
                              기준에만_못_미침=("slow_only", "sum"))
g["못_푼_비율"] = 1 - g.푼_seed / g.전체
print(g.to_string())
print()
print("판정에 쓸 것: 「못 푼」이 모두 늘 0 을 내는 해라면 국소 최소에 갇힌 것이고,")
print("             기준에만 못 미친 것이 섞여 있다면 상한이 짧았을 뿐일 수 있다.")

## C. 가속 항 α 를 빼면 얼마나 느려지는가

식 (9) 의 α 가 하는 일을 훑기 수로 잰다.

- **H1**: α = 0.9 가 α = 0 보다 적은 훑기로 푼다.
- **H1-a**: α = 0 에서 상한 안에 푸는 seed 가 거의 없다면 「몇 배 빠르다」로 적을 수 없다. **상한에 걸린 행을 그대로 남긴다.**

In [ ]:
SEEDS_C = 30
rows = []
t0 = time.time()
for alpha in (0.0, 0.5, 0.9):
    for s in range(SEEDS_C):
        r = train(SYM_X, SYM_D, [6, 2, 1], seed=s, eps=0.1, alpha=alpha, sweeps=MAX_SWEEPS)
        rows.append(dict(alpha=alpha, seed=s, converged=r["converged"],
                         sweeps=r["first_solved"] or MAX_SWEEPS, hit_cap=not r["converged"],
                         E=r["E"], max_sweeps=MAX_SWEEPS))
C = save(pd.DataFrame(rows), "C_alpha.csv")
print("%.1f초" % (time.time() - t0))

print()
gc = C.groupby("alpha").agg(푼_seed=("converged", "sum"), 전체=("seed", "count"),
                            푼것의_훑기_중앙값=("sweeps", lambda s: np.median(s)),
                            상한에_걸린_행=("hit_cap", "sum"))
print(gc.to_string())
print()
for a, g in C.groupby("alpha"):
    okg = g[g.converged]
    print("  α=%.1f  푼 seed 의 훑기 중앙값 %s"
          % (a, ("%.0f" % okg.sweeps.median()) if len(okg) else "푼 seed 가 없다"))

## D. 가족 트리

논문 그림 2 의 두 집안을 코드로 세운다. 관계는 열두 가지이고, 두 집안은 이름만 다르고 구조가 같다.

**먼저 개수를 맞춰 본다.** 논문은 「104 개의 가능한 세 낱말 조합」이라고 적었다.
⟨사람 1⟩⟨관계⟩⟨사람 2⟩ 를 전부 세면 몇 개인지, 그리고 ⟨사람 1⟩⟨관계⟩ 짝이 몇 개인지를 나누어 센다.

In [ ]:
MARRIAGES = [("Christopher", "Penelope"), ("Andrew", "Christine"), ("Margaret", "Arthur"),
             ("Victoria", "James"), ("Jennifer", "Charles"),
             ("Roberto", "Maria"), ("Pierro", "Francesca"), ("Gina", "Emilio"),
             ("Lucia", "Marco"), ("Angela", "Tomaso")]
CHILDREN = {("Christopher", "Penelope"): ["Arthur", "Victoria"],
            ("Andrew", "Christine"): ["James", "Jennifer"],
            ("Victoria", "James"): ["Colin", "Charlotte"],
            ("Roberto", "Maria"): ["Emilio", "Lucia"],
            ("Pierro", "Francesca"): ["Marco", "Angela"],
            ("Lucia", "Marco"): ["Alfonso", "Sophia"]}
MALE = set("Christopher Andrew Arthur James Charles Colin "
           "Roberto Pierro Emilio Marco Tomaso Alfonso".split())
ENG = "Christopher Penelope Andrew Christine Margaret Arthur Victoria James Jennifer Charles Colin Charlotte".split()
ITA = "Roberto Maria Pierro Francesca Gina Emilio Lucia Marco Angela Tomaso Alfonso Sophia".split()
PEOPLE = ENG + ITA

spouse = {}
for a, b in MARRIAGES:
    spouse[a] = b
    spouse[b] = a
parents = {}
for (p1, p2), kids in CHILDREN.items():
    father_, mother_ = (p1, p2) if p1 in MALE else (p2, p1)
    for k in kids:
        parents[k] = (father_, mother_)


def kids_of(p):
    out = []
    for (a, b), kids in CHILDREN.items():
        if p in (a, b):
            out += kids
    return out


def sibs(p):
    return [] if p not in parents else [c for c in kids_of(parents[p][0]) if c != p]


def uncles(p):
    out = set()
    for par in parents.get(p, ()):
        for s in sibs(par):
            if s in MALE:
                out.add(s)
            elif s in spouse:
                out.add(spouse[s])
    return sorted(out)


def aunts(p):
    out = set()
    for par in parents.get(p, ()):
        for s in sibs(par):
            if s not in MALE:
                out.add(s)
            elif s in spouse:
                out.add(spouse[s])
    return sorted(out)


REL = {
    "father":   lambda p: [parents[p][0]] if p in parents else [],
    "mother":   lambda p: [parents[p][1]] if p in parents else [],
    "husband":  lambda p: [spouse[p]] if p in spouse and spouse[p] in MALE else [],
    "wife":     lambda p: [spouse[p]] if p in spouse and spouse[p] not in MALE else [],
    "son":      lambda p: [c for c in kids_of(p) if c in MALE],
    "daughter": lambda p: [c for c in kids_of(p) if c not in MALE],
    "uncle":    uncles,
    "aunt":     aunts,
    "brother":  lambda p: [s for s in sibs(p) if s in MALE],
    "sister":   lambda p: [s for s in sibs(p) if s not in MALE],
    "nephew":   lambda p: sorted([q for q in PEOPLE if q in MALE and p in uncles(q) + aunts(q)]),
    "niece":    lambda p: sorted([q for q in PEOPLE if q not in MALE and p in uncles(q) + aunts(q)]),
}
REL_NAMES = list(REL.keys())

answers = {}
n_triples = 0
for p in PEOPLE:
    for r in REL_NAMES:
        qs = REL[r](p)
        if qs:
            answers[(p, r)] = sorted(qs)
            n_triples += len(qs)
PAIRS = sorted(answers)

print("사람 %d 명, 관계 %d 가지" % (len(PEOPLE), len(REL_NAMES)))
print("세 낱말 조합 ⟨사람1⟩⟨관계⟩⟨사람2⟩ 의 개수 :", n_triples)
print("⟨사람1⟩⟨관계⟩ 짝의 개수                  :", len(PAIRS), "   <- 논문이 적은 104 와 견줄 수")
print("답이 둘인 짝                              :", sum(1 for k in PAIRS if len(answers[k]) > 1))
print()
print("Colin 의 aunt :", REL["aunt"]("Colin"), "   (논문 그림 3 의 예시와 견준다)")
print("Colin 의 uncle:", REL["uncle"]("Colin"))

In [ ]:
# 입력 36 (사람 24 + 관계 12), 출력 24. 사람 한 명에 유닛 하나씩 쓰는 국소 표현이다.
pidx = {p: i for i, p in enumerate(PEOPLE)}
ridx = {r: i for i, r in enumerate(REL_NAMES)}
FX = np.zeros((len(PAIRS), 36))
FD = np.zeros((len(PAIRS), 24))
for n, (p, r) in enumerate(PAIRS):
    FX[n, pidx[p]] = 1.0
    FX[n, 24 + ridx[r]] = 1.0
    for q in answers[(p, r)]:
        FD[n, pidx[q]] = 1.0

FAM_SIZES = [36, 12, 12, 6, 24]          # 논문 그림 3: (6+6) -> 12 -> 6 -> 24
MASK0 = np.zeros((36, 12))
MASK0[:24, :6] = 1.0                      # 사람 24 -> 왼쪽 6 유닛
MASK0[24:, 6:] = 1.0                      # 관계 12 -> 오른쪽 6 유닛
print("입력", FX.shape, " 출력", FD.shape, " 층", FAM_SIZES)
print("첫 층에서 살아 있는 연결", int(MASK0.sum()), "/", MASK0.size)


def fam_run(seed, sweeps, eps_late, alpha_late=0.9, decay=0.002, margin=(0.8, 0.2), n_test=4):
    """학습 짝을 무작위로 n_test 개 빼 두고 나머지로 학습한다.
    논문의 일정: 처음 20 훑기는 ε=0.005, α=0.5, 그 뒤 ε, α 를 바꾼다."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(PAIRS))
    te, tr = idx[:n_test], idx[n_test:]
    sched = lambda t: (0.005, 0.5) if t <= 20 else (eps_late, alpha_late)
    r = train(FX[tr], FD[tr], FAM_SIZES, seed=seed, sweeps=sweeps, decay=decay,
              margin=margin, mask0=MASK0, schedule=sched, stop_when_solved=False)

    def score(ix):
        Y = forward(r["W"], r["b"], FX[ix])[-1]
        strict = int(solved(Y, FD[ix]).sum())
        top = 0
        for row in range(len(ix)):
            k = int(FD[ix][row].sum())
            if set(np.argsort(-Y[row])[:k]) == set(np.flatnonzero(FD[ix][row])):
                top += 1
        return strict, top, len(ix)

    return r, score(tr), score(te)

### D-1. 논문의 설정값 그대로

ε = 0.01, α = 0.9 (처음 20 훑기만 0.005 / 0.5), 1,500 훑기, 가중치 감쇠 0.2%, 오차 0 처리 켬.

**재는 값 둘.** `엄격` 은 24 개 출력이 모두 0.8 위 · 0.2 아래인 예의 수이고,
`최상위` 는 정답 개수만큼 가장 활성이 높은 출력이 정답과 정확히 같은 예의 수다.
**둘 다 적는다 — 논문은 「맞혔다」의 기준을 적지 않았다.**

- **H1**: 학습에 쓴 100 개를 대부분 맞히고, 빼 둔 4 개도 맞힌다 (논문은 4 개를 맞혔다고 적었다).
- **H1-a**: 1,500 훑기 안에 학습 자체가 되지 않으면 그대로 적는다. 그때는 D-2 의 격자로 넘어간다.

In [ ]:
SEEDS_D1 = 10
rows = []
t0 = time.time()
for s in range(SEEDS_D1):
    r, tr_s, te_s = fam_run(s, sweeps=1500, eps_late=0.01, decay=0.002, margin=(0.8, 0.2))
    rows.append(dict(seed=s, sweeps=1500, eps_late=0.01, alpha_late=0.9, decay=0.002,
                     margin="켬", E=r["E"],
                     train_strict=tr_s[0], train_top=tr_s[1], train_n=tr_s[2],
                     test_strict=te_s[0], test_top=te_s[1], test_n=te_s[2]))
D1 = save(pd.DataFrame(rows), "D1_family_paper_settings.csv")
print("%.1f초" % (time.time() - t0))

print()
print("논문 설정 그대로, seed %d 개" % SEEDS_D1)
print("  학습 %d 개 중 — 엄격 평균 %.1f, 최상위 평균 %.1f"
      % (D1.train_n.iloc[0], D1.train_strict.mean(), D1.train_top.mean()))
print("  시험 %d 개 중 — 엄격 합계 %d/%d, 최상위 합계 %d/%d"
      % (D1.test_n.iloc[0], D1.test_strict.sum(), D1.test_n.sum(),
         D1.test_top.sum(), D1.test_n.sum()))

### D-2. ε 를 바꿔 가며 — 어디서 학습이 되는가

D-1 이 풀리지 않을 경우를 대비해 격자를 미리 정해 둔다. **격자는 돌리기 전에 정한 것이고 결과를 보고 넓히지 않는다.**

ε ∈ {0.01, 0.02, 0.05, 0.1} × 감쇠 ∈ {0, 0.002} × 오차 0 처리 ∈ {켬, 끔} = 16 칸, seed 8 개, 6,000 훑기.

In [ ]:
SEEDS_D2 = 8
SWEEPS_D2 = 6000
rows = []
t0 = time.time()
for eps_late in (0.01, 0.02, 0.05, 0.1):
    for decay in (0.0, 0.002):
        for margin in ((0.8, 0.2), None):
            for s in range(SEEDS_D2):
                r, tr_s, te_s = fam_run(s, sweeps=SWEEPS_D2, eps_late=eps_late,
                                        decay=decay, margin=margin)
                rows.append(dict(eps_late=eps_late, decay=decay,
                                 margin="켬" if margin else "끔", seed=s,
                                 sweeps=SWEEPS_D2, E=r["E"],
                                 train_strict=tr_s[0], train_top=tr_s[1], train_n=tr_s[2],
                                 test_strict=te_s[0], test_top=te_s[1], test_n=te_s[2]))
        print("  eps %.2f 감쇠 %.3f 끝  (%.0f초)" % (eps_late, decay, time.time() - t0))
D2 = save(pd.DataFrame(rows), "D2_family_grid.csv")
print("%.1f초" % (time.time() - t0))

g2 = D2.groupby(["eps_late", "decay", "margin"]).agg(
    학습_엄격=("train_strict", "mean"), 학습_최상위=("train_top", "mean"),
    시험_엄격=("test_strict", "sum"), 시험_최상위=("test_top", "sum"),
    시험_전체=("test_n", "sum")).reset_index()
print()
print(g2.to_string(index=False))
BEST = g2.sort_values("학습_최상위", ascending=False).iloc[0]
print()
print("학습 성적이 가장 좋은 칸:", dict(BEST[["eps_late", "decay", "margin"]]))

### D-3. 가중치 감쇠를 빼면 은닉 유닛의 축이 덜 읽히는가

논문은 감쇠를 **가중치를 해석하기 쉽게 하려고** 넣었다고 적었고, 그것이 성능에 미친 영향은 적지 않았다.
2 층의 사람 쪽 6 유닛이 24 명에게 준 가중치를 놓고, **국적**(영국 / 이탈리아)과 **세대**(1 / 2 / 3)가
얼마나 또렷하게 갈리는지를 잰다.

재는 값은 **η²** 다. 한 유닛의 24 개 가중치에서 *묶음 사이의 분산 / 전체 분산* 이고, 1 에 가까울수록 그 묶음을 또렷하게 가른다.
작은 예시로, 영국 12 명에 모두 +1, 이탈리아 12 명에 모두 -1 을 주면 η² = 1 이다.

- **H1**: 감쇠를 켠 쪽의 η² 최댓값이 더 크다 (축이 더 또렷하다).
- **H1-a**: 차이가 seed 사이의 변동 폭보다 작으면 순서를 매기지 않는다.

In [ ]:
def gen_of(p):
    """부모를 따라 올라가며 세대를 센다. 부모가 없으면 None 을 돌려준다."""
    if p not in parents:
        return None
    up = gen_of(parents[p][0])
    return 2 if up is None else up + 1


GEN = {}
for p in PEOPLE:                       # 부모가 있는 사람부터
    g = gen_of(p)
    if g is not None:
        GEN[p] = g
for p in PEOPLE:                       # 부모가 없는 사람은 배우자의 세대를 따르고, 없으면 1 세대다
    if p not in GEN:
        GEN[p] = GEN.get(spouse.get(p), 1)
NAT = np.array([0 if p in ENG else 1 for p in PEOPLE])
GENV = np.array([GEN[p] for p in PEOPLE])
print("세대 분포:", {g: int((GENV == g).sum()) for g in sorted(set(GENV))})


def eta2(values, groups):
    """묶음 사이의 분산 / 전체 분산."""
    values = np.asarray(values, dtype=float)
    tot = values.var()
    if tot <= 1e-12:
        return 0.0
    between = 0.0
    for g in np.unique(groups):
        m = groups == g
        between += m.sum() * (values[m].mean() - values.mean()) ** 2
    return float(between / len(values) / tot)


SEEDS_D3 = 12

# 감쇠를 켠 쪽과 끈 쪽을 견주려면 양쪽 모두 학습이 되는 설정이어야 한다.
# 한쪽만 학습되는 설정에서 축의 또렷함을 견주면 「감쇠의 효과」가 아니라 「학습이 되었는가」를 재게 된다.
# 그래서 감쇠 수준마다 D-2 격자에서 학습 성적이 가장 좋은 칸을 따로 고른다.
BEST_BY_DECAY = {}
for dv, gg in g2.groupby("decay"):
    BEST_BY_DECAY[float(dv)] = gg.sort_values("학습_최상위", ascending=False).iloc[0]
for dv, row in sorted(BEST_BY_DECAY.items()):
    print("감쇠 %.3f 에서 가장 좋은 칸: eps %.2f, 오차0처리 %s  (학습 최상위 %.1f)"
          % (dv, row["eps_late"], row["margin"], row["학습_최상위"]))
print()

rows = []
t0 = time.time()
for decay in (0.002, 0.0):
    row_b = BEST_BY_DECAY[decay]
    eps_best = float(row_b["eps_late"])
    margin_best = (0.8, 0.2) if row_b["margin"] == "켬" else None
    for s in range(SEEDS_D3):
        r, tr_s, te_s = fam_run(s, sweeps=SWEEPS_D2, eps_late=eps_best,
                                decay=decay, margin=margin_best)
        Wp = r["W"][0][:24, :6]           # 사람 24 -> 2 층의 6 유닛
        nat = [eta2(Wp[:, u], NAT) for u in range(6)]
        gen = [eta2(Wp[:, u], GENV) for u in range(6)]
        rows.append(dict(decay=decay, seed=s, eps_late=eps_best,
                         margin=row_b["margin"], sweeps=SWEEPS_D2,
                         nat_eta2_max=max(nat), gen_eta2_max=max(gen),
                         nat_eta2_all=";".join("%.3f" % v for v in nat),
                         gen_eta2_all=";".join("%.3f" % v for v in gen),
                         weight_abs_mean=float(np.abs(Wp).mean()),
                         train_top=tr_s[1], test_top=te_s[1], test_n=te_s[2]))
D3 = save(pd.DataFrame(rows), "D3_family_decay.csv")
print("%.1f초" % (time.time() - t0))

print()
g3 = D3.groupby("decay").agg(eps=("eps_late", "first"),
                             국적_eta2_중앙값=("nat_eta2_max", "median"),
                             국적_eta2_최소=("nat_eta2_max", "min"),
                             국적_eta2_최대=("nat_eta2_max", "max"),
                             세대_eta2_중앙값=("gen_eta2_max", "median"),
                             가중치크기_평균=("weight_abs_mean", "mean"),
                             학습_최상위=("train_top", "mean"))
print(g3.to_string())
print()
print("seed 사이의 변동 폭(최대-최소)이 두 조건의 차이보다 크면 순서를 매기지 않는다.")
print("양쪽의 학습 최상위 성적이 크게 다르면 η² 의 차이를 감쇠의 효과라고 부를 수 없다 — 그것도 함께 본다.")

# 나중에 논문 그림 4 처럼 그리려고 가중치 자체도 남긴다 (학습이 가장 잘 된 칸, seed 0)
row_top = max(BEST_BY_DECAY.values(), key=lambda r: r["학습_최상위"])
r0, _, _ = fam_run(0, sweeps=SWEEPS_D2, eps_late=float(row_top["eps_late"]),
                   decay=float(row_top["decay"]),
                   margin=(0.8, 0.2) if row_top["margin"] == "켬" else None)
print("그림으로 남기는 조건: eps %.2f, 감쇠 %.3f, 오차0처리 %s"
      % (row_top["eps_late"], row_top["decay"], row_top["margin"]))
Wp0 = r0["W"][0][:24, :6]
save(pd.DataFrame(Wp0, index=PEOPLE, columns=["unit%d" % (i + 1) for i in range(6)])
     .reset_index().rename(columns={"index": "person"})
     .assign(nationality=["영국" if p in ENG else "이탈리아" for p in PEOPLE],
             generation=[GEN[p] for p in PEOPLE]),
     "D3_person_unit_weights.csv")

## E. 층이 깊어지면 층별 기울기가 몇 배씩 줄어드는가

노트에 적어 둔 것은 **식 (5) 의 $y(1-y)$ 가 최대 0.25 라 층마다 곱해지면 줄어든다**는 것이었다.
실제로는 여기에 가중치의 크기도 곱해지므로, 한 층마다 줄어드는 배수를 직접 잰다.

- **H1**: 아래층으로 갈수록 $|\partial E/\partial w|$ 의 평균이 일정한 배수로 줄어든다.
- **재는 것**: 학습 시작 시점(난수 가중치)의 층별 평균 크기와, 이웃한 두 층 사이의 배수.

In [ ]:
def layer_grads(depth, width=8, n=64, seed=0):
    """은닉층 depth 개짜리 망에서 학습 시작 시점의 층별 |dE/dw| 평균을 돌려준다."""
    rng = np.random.default_rng(seed)
    sizes = [width] * (depth + 1) + [1]
    W, b = init_net(sizes, rng)
    X = rng.integers(0, 2, (n, width)).astype(float)
    D = rng.integers(0, 2, (n, 1)).astype(float)
    ys = forward(W, b, X)
    gW, _ = backward(W, ys, D)
    return [float(np.abs(g).mean()) for g in gW]


rows = []
for depth in (1, 2, 3, 4, 5, 6, 7):
    for s in range(20):
        vals = layer_grads(depth, seed=s)
        for L, v in enumerate(vals):
            rows.append(dict(depth=depth, seed=s, layer=L, n_layers=len(vals), grad_abs_mean=v))
E = save(pd.DataFrame(rows), "E_depth.csv")

print()
for depth, g in E.groupby("depth"):
    m = g.groupby("layer").grad_abs_mean.mean()
    ratio = (m.iloc[-1] / m.iloc[0]) if m.iloc[0] > 0 else np.inf
    per = ratio ** (1.0 / max(1, len(m) - 1))
    print("은닉층 %d 개  맨 아래 %.3e  맨 위 %.3e  전체 %.3g 배  한 층마다 %.2f 배"
          % (depth, m.iloc[0], m.iloc[-1], ratio, per))
print()
print("(0.25 는 y(1-y) 의 최댓값이다. 한 층마다의 배수가 그보다 큰지 작은지를 본다.)")

## F. XOR — 퍼셉트론이 못 풀던 것

같은 절차로 은닉 유닛 둘짜리 망을 학습시킨다. 퍼셉트론 노트북의 실험 A 와 짝이 되는 자리다.

- **H1**: 대부분의 seed 에서 풀린다. **H1-a**: 풀리지 않는 seed 가 있다면 그 비율을 적는다 (국소 최소다).

In [ ]:
XOR_X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
XOR_D = np.array([[0.0], [1.0], [1.0], [0.0]])
SEEDS_F = 100

rows = []
t0 = time.time()
for s in range(SEEDS_F):
    r = train(XOR_X, XOR_D, [2, 2, 1], seed=s, eps=0.1, alpha=0.9, sweeps=MAX_SWEEPS)
    rows.append(dict(seed=s, hidden=2, converged=r["converged"],
                     sweeps=r["first_solved"] or MAX_SWEEPS, E=r["E"], max_sweeps=MAX_SWEEPS))
for s in range(SEEDS_F):
    r = train(XOR_X, XOR_D, [2, 4, 1], seed=s, eps=0.1, alpha=0.9, sweeps=MAX_SWEEPS)
    rows.append(dict(seed=s, hidden=4, converged=r["converged"],
                     sweeps=r["first_solved"] or MAX_SWEEPS, E=r["E"], max_sweeps=MAX_SWEEPS))
F = save(pd.DataFrame(rows), "F_xor.csv")
print("%.1f초" % (time.time() - t0))

print()
print(F.groupby("hidden").agg(푼_seed=("converged", "sum"), 전체=("seed", "count"),
                              훑기_중앙값=("sweeps", "median")).to_string())

## 그림

한글 글꼴이 없는 환경에서 네모로 깨지는 것을 피하려고 **축 라벨은 영문으로 둔다.**

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = []

# 1) 대칭 검출 — 은닉 유닛 수와 푼 비율
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.6), dpi=140)
gg = Bdf.groupby("hidden")
ax[0].bar([str(h) for h in gg.groups], [1 - g.converged.mean() for _, g in gg], color="#B4552B")
ax[0].set_xlabel("hidden units"); ax[0].set_ylabel("fraction not solved")
ax[0].set_title("does adding units help escape?")
for h, g in gg:
    okg = g[g.converged]
    ax[1].scatter(np.full(len(okg), h) + np.random.uniform(-.15, .15, len(okg)),
                  okg.sweeps, s=12, alpha=.6)
ax[1].set_yscale("log"); ax[1].set_xlabel("hidden units"); ax[1].set_ylabel("sweeps to solve (log)")
ax[1].axhline(1425, color="#2E6B8A", ls="--", lw=1, label="paper: 1,425")
ax[1].legend(fontsize=8); ax[1].set_title("sweeps needed")
fig.tight_layout()
p = os.path.join(FIGURES, "A_B_symmetry.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

# 2) 가속 항
fig, ax = plt.subplots(figsize=(5.6, 3.6), dpi=140)
for a, g in C.groupby("alpha"):
    okg = g[g.converged]
    ax.scatter(np.full(len(okg), a) + np.random.uniform(-.02, .02, len(okg)), okg.sweeps,
               s=16, alpha=.7, label="alpha=%.1f (solved %d/%d)" % (a, len(okg), len(g)))
ax.set_yscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("sweeps to solve (log)")
ax.set_title("equation (9): the acceleration term")
ax.legend(fontsize=8)
fig.tight_layout()
p = os.path.join(FIGURES, "C_alpha.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

# 3) 깊이별 기울기
fig, ax = plt.subplots(figsize=(5.6, 4.0), dpi=140)
for depth, g in E.groupby("depth"):
    m = g.groupby("layer").grad_abs_mean.mean()
    ax.plot(range(len(m)), m.values, marker="o", ms=3, label="%d hidden" % depth)
ax.set_yscale("log"); ax.set_xlabel("layer (0 = closest to input)")
ax.set_ylabel("mean |dE/dw| at start (log)")
ax.set_title("the gradient shrinks going down")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
p = os.path.join(FIGURES, "E_depth.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

# 4) 논문 그림 4 처럼 — 사람 24 명에 대한 2 층 6 유닛의 가중치
fig, ax = plt.subplots(figsize=(8.2, 3.2), dpi=140)
im = ax.imshow(Wp0.T, cmap="RdBu_r", aspect="auto",
               vmin=-np.abs(Wp0).max(), vmax=np.abs(Wp0).max())
ax.set_yticks(range(6)); ax.set_yticklabels(["unit %d" % (i + 1) for i in range(6)], fontsize=8)
ax.set_xticks(range(24)); ax.set_xticklabels(PEOPLE, rotation=90, fontsize=6)
ax.axvline(11.5, color="k", lw=1)
ax.set_title("weights from the 24 people to the 6 units (left of line: English)")
fig.colorbar(im, ax=ax, shrink=.8)
fig.tight_layout()
p = os.path.join(FIGURES, "D3_person_units.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

print("그림", len(FIG), "장")
for p in FIG:
    print("  ", os.path.basename(p))

## 요약 — 아래 출력을 그대로 `README.md` 의 결과 칸에 옮긴다

In [ ]:
print("=" * 82)
print("역전파 (1986) 실험 요약   run_id =", RUN_ID)
print("=" * 82)
okA = A[A.converged]
print()
print("[A] 대칭 검출, seed %d, 훑기 상한 %d" % (len(A), MAX_SWEEPS))
print("    푼 seed %d/%d" % (len(okA), len(A)))
if len(okA):
    print("    훑기 중앙값 %.0f (최소 %d, 최대 %d)  — 논문은 1,425" % (
        okA.sweeps.median(), okA.sweeps.min(), okA.sweeps.max()))
    print("    부호 반대칭 지표 중앙값 %.4f, 최대 %.4f  (0 이면 완전한 반대칭)" % (
        okA[["anti_h1", "anti_h2"]].median().mean(), okA[["anti_h1", "anti_h2"]].max().max()))
    print("    위쪽 절반 세 크기의 비 중앙값 %.2f : %.2f : %.2f  — 논문의 그림 1 은 1 : 2 : 4" % (
        okA.r1.median(), okA.r2.median(), okA.r3.median()))
print("    못 푼 seed 의 오차 E: %s  (늘 0 을 내는 해는 4.0)" % sorted(np.round(A[~A.converged].E.values, 2)))
print()
print("[B] 은닉 유닛 수  (못 푼 것을 「늘 0 을 내는 해」와 「기준에만 못 미침」으로 가름)")
for h, g in Bdf.groupby("hidden"):
    print("    은닉 %d 개  푼 seed %2d/%2d  못 푼 비율 %.2f  그중 늘 0 %2d / 기준만 %2d  훑기 중앙값 %6.0f" % (
        h, g.converged.sum(), len(g), 1 - g.converged.mean(),
        g.stuck_all_off.sum(), g.slow_only.sum(), g.sweeps.median()))
print()
print("[C] 가속 항")
for a, g in C.groupby("alpha"):
    okg = g[g.converged]
    print("    alpha %.1f  푼 seed %2d/%2d  훑기 중앙값 %s" % (
        a, len(okg), len(g), ("%.0f" % okg.sweeps.median()) if len(okg) else "푼 seed 없음"))
print()
print("[D] 가족 트리")
print("    세 낱말 조합 %d 개, ⟨사람1⟩⟨관계⟩ 짝 %d 개, 답이 둘인 짝 %d 개" % (
    n_triples, len(PAIRS), sum(1 for k in PAIRS if len(answers[k]) > 1)))
print("    D-1 논문 설정 그대로 (1,500 훑기, eps 0.01, 감쇠 0.002, 오차0처리 켬), seed %d" % len(D1))
print("        학습 %d 개 중 엄격 평균 %.1f / 최상위 평균 %.1f" % (
    D1.train_n.iloc[0], D1.train_strict.mean(), D1.train_top.mean()))
print("        시험 엄격 %d/%d, 최상위 %d/%d" % (
    D1.test_strict.sum(), D1.test_n.sum(), D1.test_top.sum(), D1.test_n.sum()))
print("    D-2 격자에서 학습 성적이 가장 좋은 칸: eps %.2f, 감쇠 %.3f, 오차0처리 %s" % (
    BEST["eps_late"], BEST["decay"], BEST["margin"]))
print("        학습 엄격 %.1f / 최상위 %.1f, 시험 엄격 %d/%d, 최상위 %d/%d" % (
    BEST["학습_엄격"], BEST["학습_최상위"], BEST["시험_엄격"], BEST["시험_전체"],
    BEST["시험_최상위"], BEST["시험_전체"]))
print("    D-3 가중치 감쇠와 은닉 축 (감쇠 수준마다 학습 성적이 가장 좋은 칸에서, seed %d 개)" % SEEDS_D3)
for d, g in D3.groupby("decay"):
    print("        감쇠 %.3f (eps %.2f, 오차0처리 %s)  학습 최상위 %.1f" % (
        d, g.eps_late.iloc[0], g.margin.iloc[0], g.train_top.mean()))
    print("            국적 eta2 중앙값 %.3f (범위 %.3f~%.3f)  세대 eta2 중앙값 %.3f  가중치 크기 %.2f" % (
        g.nat_eta2_max.median(), g.nat_eta2_max.min(), g.nat_eta2_max.max(),
        g.gen_eta2_max.median(), g.weight_abs_mean.mean()))
print()
print("[E] 깊이별 기울기 (학습 시작 시점)")
for depth, g in E.groupby("depth"):
    m = g.groupby("layer").grad_abs_mean.mean()
    per = (m.iloc[-1] / m.iloc[0]) ** (1.0 / max(1, len(m) - 1))
    print("    은닉 %d 개  맨 아래 %.2e  맨 위 %.2e  한 층마다 %.2f 배" % (
        depth, m.iloc[0], m.iloc[-1], per))
print()
print("[F] XOR")
for h, g in F.groupby("hidden"):
    print("    은닉 %d 개  푼 seed %3d/%3d  훑기 중앙값 %.0f" % (
        h, g.converged.sum(), len(g), g.sweeps.median()))
print()
print("결과 파일")
for f in sorted(os.listdir(RESULTS)):
    print("   results/%s" % f)
for f in sorted(os.listdir(FIGURES)):
    print("   figures/%s" % f)
print("=" * 82)